In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# ==========================================
# 1. Core Transformer Architecture (Bidirectional)
# ==========================================
class BidirectionalTransformerLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        # self.self_attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.norm1 = nn.LayerNorm(d_model)
        
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model)
        )
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        # Full bidirectional attention (no causal mask)
        # attn_out, _ = self.self_attn(x, x, x)
        batch_size, seq_len, _ = x.size()
        q = self.q_proj(x).view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)

        # This automatically triggers FlashAttention-2 on H100 GPUs!
        attn_out = F.scaled_dot_product_attention(q, k, v, is_causal=False)

        # Reshape back to original
        attn_out = attn_out.transpose(1, 2).contiguous().view(batch_size, seq_len, -1)
        attn_out = self.out_proj(attn_out)
        x = self.norm1(x + attn_out)
        mlp_out = self.mlp(x)
        x = self.norm2(x + mlp_out)
        return x

In [ ]:
# ==========================================
# 2. TRM Diffusion Model Setup
# ==========================================
class TRM_Diffusion(nn.Module):
    def __init__(self, vocab_size, d_model=512, n_heads=8, d_ff=2048, num_layers=2, mask_token_id=-100, n = 6):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.mask_token_id = mask_token_id
        self.n = n
        
        # Embeddings
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.mask_emb = nn.Parameter(torch.randn(1, 1, d_model)) # Special embedding for MASK
        self.pos_emb = nn.Embedding(4096, d_model) # Assuming max sequence length 4096
        
        # TRM "Tiny Network" Backbone
        self.layers = nn.ModuleList([
            BidirectionalTransformerLayer(d_model, n_heads, d_ff) 
            for _ in range(num_layers)
        ])
        
        # Output head to predict vocabulary
        self.output_head = nn.Linear(d_model, vocab_size)
        
        # Initialization for the latent space z
        self.z_init = nn.Parameter(torch.randn(1, 1, d_model))

        self.halting_head = nn.Sequential(
            nn.Linear(self.d_model, self.d_model // 2),
            nn.ReLU(),
            nn.Linear(self.d_model // 2, 1),
            nn.Sigmoid() # Outputs a probability between 0 and 1
        )

    def get_embeddings(self, tokens):
            # Handle regular tokens and the special MASK token
            seq_len = tokens.size(1)
            positions = torch.arange(seq_len, device=tokens.device).unsqueeze(0).expand_as(tokens)
            
            emb = self.token_emb(torch.clamp(tokens, min=0)) # Clamp prevents error on mask_token_id
            
            # Apply special mask embedding where necessary
            mask_positions = (tokens == self.mask_token_id).unsqueeze(-1)
            emb = torch.where(mask_positions, self.mask_emb, emb)
            
            return emb + self.pos_emb(positions)

    def latent_recursion(self, x_t_emb, z, n ):
            """x_t_emb is the combined prompt + masked response embeddings"""
            for _ in range(n):
                # We now add the sequence to the latent state
                combined_state = x_t_emb + z 
                
                for layer in self.layers:
                    combined_state = layer(combined_state)
                
                z = combined_state
                
            logits = self.output_head(z)
            return logits, z

    def deep_recursion(self, x_t_emb, z, T=3):
        with torch.no_grad():
            for _ in range(T - 1):
                logits, z = self.latent_recursion(x_t_emb, z, self.n)
                
        logits, z = self.latent_recursion(x_t_emb, z, self.n)
        return logits, z.detach()
    
    # And add an ACT-specific forward method to the class:
    def get_halting_prob(self, z):
        # z is shape (batch, seq_len, d_model)
        # We take the mean across the sequence to get a single 'thought state' for the whole sentence
        z_mean = z.mean(dim=1) 
        return self.halting_head(z_mean) # Returns shape (batch, 1)

In [48]:
# ==========================================
# 3. Training Logic (Masking & Deep Supervision)
# ==========================================
def train_step_conditional(model, optimizer, full_story_tokens, prompt_length=5, N_sup=16):
    model.train()
    batch_size, seq_len = full_story_tokens.size()
    
    t = torch.rand(1).item()
    if t == 0: t = 1e-5 
    
    # 1. Create a mask that ONLY applies to the response (after prompt_length)
    mask_probs = torch.zeros((batch_size, seq_len), device=full_story_tokens.device)
    mask_probs[:, prompt_length:] = t  # Only mask the response part
    
    mask_boolean = torch.bernoulli(mask_probs).bool()
    
    # 2. Create x_t (Clean prompt + Masked response)
    x_t = full_story_tokens.clone()
    x_t[mask_boolean] = model.mask_token_id
    
    # 3. Get embeddings for the unified sequence
    x_t_emb = model.get_embeddings(x_t)
    z = model.z_init.expand(batch_size, seq_len, model.d_model)
    
    optimizer.zero_grad()
    total_loss = 0.0
    
    for step in range(N_sup):
        logits, z = model.deep_recursion(x_t_emb, z)
        
        logits_flat = logits.view(-1, model.vocab_size)
        targets_flat = full_story_tokens.view(-1)
        mask_flat = mask_boolean.view(-1)
        
        # Calculate loss ONLY on the tokens that were actually masked
        loss = F.cross_entropy(
            logits_flat[mask_flat], 
            targets_flat[mask_flat], 
            reduction='mean'
        )
        
        scaled_loss = loss / t
        total_loss += scaled_loss.item()
        scaled_loss.backward()
        
    optimizer.step()
    return total_loss / N_sup

In [46]:
@torch.no_grad()
def generate_conditional(model, x_t_initial, prompt_length, sampling_steps=64):
    """
    Generates text conditionally using the LLaDA reverse diffusion process.
    
    Args:
        model: The trained TRM_Diffusion model.
        x_t_initial: Tensor of shape (batch, seq_len) containing the clean prompt 
                     followed by MASK tokens for the response.
        prompt_length: The number of tokens making up the prompt.
        sampling_steps: Number of diffusion unmasking steps.
    """
    model.eval()
    batch_size, seq_len = x_t_initial.size()
    device = x_t_initial.device
    
    # Start with the initial state (clean prompt + fully masked response)
    r_t = x_t_initial.clone()
    
    # We only scale our unmasking math based on the part of the sequence we are generating
    answer_length = seq_len - prompt_length
    
    # Initialize the latent reasoning vector z
    z = model.z_init.expand(batch_size, seq_len, model.d_model)
    
    # Create timesteps from 1.0 down to 1/N
    timesteps = torch.linspace(1.0, 1.0 / sampling_steps, sampling_steps, device=device)
    
    for t_idx, t in enumerate(timesteps):
        # Calculate the next timestep 's'
        s = t - (1.0 / sampling_steps) if t_idx < len(timesteps) - 1 else torch.tensor(0.0)
        
        # 1. Embed the current sequence (prompt + whatever is currently unmasked)
        r_t_emb = model.get_embeddings(r_t)
        
        # 2. Get predictions from the TRM backbone
        logits, z = model.deep_recursion(r_t_emb, z)
        probs = F.softmax(logits, dim=-1)
        confidences, predicted_tokens = torch.max(probs, dim=-1)
        
        # 3. Update sequence: Replace ONLY the currently masked tokens with predictions
        is_masked = (r_t == model.mask_token_id)
        r_0_hat = torch.where(is_masked, predicted_tokens, r_t)
        
        # --- Low-Confidence Remasking ---
        # Calculate how many tokens IN THE RESPONSE should remain unmasked at step s
        n_unmasked_target = int(answer_length * (1.0 - s.item()))
        num_to_remask = answer_length - n_unmasked_target
        
        if num_to_remask > 0:
            # Set confidence of already unmasked tokens to infinity.
            # *Crucial*: Because 'is_masked' is False for the prompt tokens, 
            # their confidence is set to infinity here, guaranteeing they will 
            # NEVER be selected for remasking!
            masked_confidences = confidences.masked_fill(~is_masked, float('inf'))
            
            # Find the indices of the lowest confidence tokens across the sequence
            _, indices_to_remask = torch.topk(masked_confidences, num_to_remask, dim=1, largest=False)
            
            # Remask those specific, low-confidence tokens for the next iteration
            r_t = r_0_hat.clone()
            r_t.scatter_(1, indices_to_remask, model.mask_token_id)
        else:
            # Final step: keep everything unmasked
            r_t = r_0_hat
            
    return r_t

Train the tokenizer

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
import os

# ==========================================
# 1. Train a Custom Tokenizer on TinyStories
# ==========================================
def train_custom_tokenizer(dataset, vocab_size=10000, save_path="tinystories_tokenizer.json"):
    """
    Trains a Byte-Pair Encoding (BPE) tokenizer from scratch.
    """
    if os.path.exists(save_path):
        print(f"Loading existing tokenizer from {save_path}...")
        return Tokenizer.from_file(save_path)
        
    print("Training new tokenizer from scratch. This might take a minute...")
    
    # Initialize a BPE Tokenizer
    tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
    tokenizer.pre_tokenizer = Whitespace()

    # Define our special tokens
    # [PAD] = 0, [UNK] = 1, [BOS] = 2, [EOS] = 3
    special_tokens = ["[PAD]", "[UNK]", "[BOS]", "[EOS]"]
    
    # Configure the trainer
    trainer = BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=special_tokens,
        show_progress=True
    )

    # We use a generator to avoid loading the entire dataset into RAM
    def batch_iterator(batch_size=1000):
        for i in range(0, len(dataset), batch_size):
            yield dataset[i : i + batch_size]["text"]

    # Train and save
    tokenizer.train_from_iterator(batch_iterator(), trainer=trainer)
    tokenizer.save(save_path)
    print("Tokenizer training complete!")
    
    return tokenizer

# ==========================================
# 2. PyTorch Dataset for TRM-Diffusion
# ==========================================
class TinyStoriesDiffusionDataset(Dataset):
    def __init__(self, hf_dataset, tokenizer, max_length=256):
        self.dataset = hf_dataset
        self.tokenizer = tokenizer
        self.max_length = max_length
        
        # Grab special token IDs
        self.pad_id = self.tokenizer.token_to_id("[PAD]")
        self.bos_id = self.tokenizer.token_to_id("[BOS]")
        self.eos_id = self.tokenizer.token_to_id("[EOS]")

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        # 1. Get text and encode
        text = self.dataset[idx]['text']
        encoded = self.tokenizer.encode(text).ids
        
        # 2. Add [EOS] to the end
        encoded = encoded + [self.eos_id]
        
        # 3. Truncate or Pad
        if len(encoded) > self.max_length:
            # Truncate if too long (keep EOS at the end)
            encoded = encoded[:self.max_length - 1] + [self.eos_id]
        else:
            # Pad with EOS tokens if too short 
            # (LLaDA strategy: pad with EOS so it learns to predict them!)
            padding_length = self.max_length - len(encoded)
            encoded = encoded + ([self.eos_id] * padding_length)
            
        # Suppose we want the first 5 tokens to act as the prompt
        split_idx = 5

        # The prompt is the beginning of the story
        prompt_tokens = [self.bos_id] + encoded[:split_idx]

        # The target is the rest of the story
        target_tokens = encoded[split_idx:]

        prompt = torch.tensor(prompt_tokens, dtype=torch.long)
        target = torch.tensor(target_tokens, dtype=torch.long)
        
        return prompt, target

# ==========================================
# 3. Putting it together: The Setup Function
# ==========================================
def get_dataloaders(batch_size=32, max_length=256, vocab_size=10000):
    print("Downloading/Loading TinyStories...")
    # Load the training and validation splits
    ds_train = load_dataset("roneneldan/TinyStories", split="train")
    ds_val = load_dataset("roneneldan/TinyStories", split="validation")
    
    # Train/Load Tokenizer
    tokenizer = train_custom_tokenizer(ds_train, vocab_size=vocab_size)
    
    # Create PyTorch Datasets
    train_dataset = TinyStoriesDiffusionDataset(ds_train, tokenizer, max_length=max_length)
    val_dataset = TinyStoriesDiffusionDataset(ds_val, tokenizer, max_length=max_length)
    
    # Create DataLoaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, drop_last=True)
    
    return train_loader, val_loader, tokenizer

# ==========================================
# Usage Example
# ==========================================
if __name__ == "__main__":
    # Hyperparameters
    BATCH_SIZE = 16
    MAX_SEQ_LENGTH = 256
    VOCAB_SIZE = 10000  # Smaller vocab size is great for tiny models!
    
    train_loader, val_loader, tokenizer = get_dataloaders(
        batch_size=BATCH_SIZE, 
        max_length=MAX_SEQ_LENGTH, 
        vocab_size=VOCAB_SIZE
    )
    
    # Test a single batch
    for prompts, targets in train_loader:
        print(f"Prompt shape (BOS token): {prompts.shape}")
        print(f"Targets shape (Story): {targets.shape}")
        
        # Decode the first sequence in the batch just to verify
        print("\nDecoded target 0:")
        # We replace EOS id with string just to visualize the padding
        decoded_words = [tokenizer.id_to_token(id) for id in targets[0].tolist()]
        print(" ".join(decoded_words[:50]) + " ...")
        break

c:\Users\MSI-1\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Downloading/Loading TinyStories...
Training new tokenizer from scratch. This might take a minute...
Tokenizer training complete!
Prompt shape (BOS token): torch.Size([16, 1])
Targets shape (Story): torch.Size([16, 256])

Decoded target 0:
Once there was a young girl called Lucy . One day , Lucy went to the park with her mom . When she arrived , she saw a big , long balloon . It was bright and colourful and she wanted to have it . She ran to it and ...


In [33]:
# ==========================================
# 2. PyTorch Dataset for TRM-Diffusion
# ==========================================
class TinyStoriesDiffusionDataset(Dataset):
    def __init__(self, hf_dataset, tokenizer, max_length=256):
        self.dataset = hf_dataset
        self.tokenizer = tokenizer
        self.max_length = max_length
        
        # Grab special token IDs
        self.pad_id = self.tokenizer.token_to_id("[PAD]")
        self.bos_id = self.tokenizer.token_to_id("[BOS]")
        self.eos_id = self.tokenizer.token_to_id("[EOS]")

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        # 1. Get text and encode
        text = self.dataset[idx]['text']
        encoded = self.tokenizer.encode(text).ids
        
        # 2. Add [EOS] to the end
        encoded = encoded + [self.eos_id]
        
        # 3. Truncate or Pad
        if len(encoded) > self.max_length:
            # Truncate if too long (keep EOS at the end)
            encoded = encoded[:self.max_length - 1] + [self.eos_id]
        else:
            # Pad with EOS tokens if too short 
            # (LLaDA strategy: pad with EOS so it learns to predict them!)
            padding_length = self.max_length - len(encoded)
            encoded = encoded + ([self.eos_id] * padding_length)
            
        # Suppose we want the first 5 tokens to act as the prompt
        split_idx = 5

        # The prompt is the beginning of the story
        prompt_tokens = [self.bos_id] + encoded[:split_idx]

        # The target is the rest of the story
        target_tokens = encoded[split_idx:]

        prompt = torch.tensor(prompt_tokens, dtype=torch.long)
        target = torch.tensor(target_tokens, dtype=torch.long)
        
        return prompt, target

# ==========================================
# 3. Putting it together: The Setup Function
# ==========================================
def get_dataloaders(batch_size=32, max_length=256, vocab_size=10000):
    print("Downloading/Loading TinyStories...")
    # Load the training and validation splits
    ds_train = load_dataset("roneneldan/TinyStories", split="train")
    ds_val = load_dataset("roneneldan/TinyStories", split="validation")
    
    # Train/Load Tokenizer
    tokenizer = Tokenizer.from_file("C:/Users/MSI-1/Desktop/adnan_final/tinystories_tokenizer.json")
    
    # Create PyTorch Datasets
    train_dataset = TinyStoriesDiffusionDataset(ds_train, tokenizer, max_length=max_length)
    val_dataset = TinyStoriesDiffusionDataset(ds_val, tokenizer, max_length=max_length)
    
    # Create DataLoaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, drop_last=True)
    
    return train_loader, val_loader, tokenizer

# ==========================================
# Usage Example
# ==========================================
if __name__ == "__main__":
    # Hyperparameters
    BATCH_SIZE = 16
    MAX_SEQ_LENGTH = 256
    VOCAB_SIZE = 10000  # Smaller vocab size is great for tiny models!
    
    train_loader, val_loader, tokenizer = get_dataloaders(
        batch_size=BATCH_SIZE, 
        max_length=MAX_SEQ_LENGTH, 
        vocab_size=VOCAB_SIZE
    )
    
    # Test a single batch
    for prompts, targets in train_loader:
        print(f"Prompt shape (BOS token): {prompts.shape}")
        print(f"Targets shape (Story): {targets.shape}")
        
        # Decode the first sequence in the batch just to verify
        print("\nDecoded target 0:")
        # We replace EOS id with string just to visualize the padding
        decoded_words = [tokenizer.id_to_token(id) for id in targets[0].tolist()]
        print(" ".join(decoded_words[:50]) + " ...")
        break

Downloading/Loading TinyStories...
Prompt shape (BOS token): torch.Size([16, 6])
Targets shape (Story): torch.Size([16, 251])

Decoded target 0:
there was a little girl called Emily who wanted to make something . She grabbed her stuff and went off into the garden . â € œWhat do I need to attach the stuff ? â € she asked her dad . â € œ Some glue should do the ...


In [ ]:
import torch
from torch.optim import AdamW
from tqdm import tqdm

# (Assume get_dataloaders, TRM_Diffusion, train_step, and generate 
# are imported or defined above this script)

def train():
    # ==========================================
    # 1. Setup Device & Hyperparameters
    # ==========================================
    device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
    print(f"Training on device: {device}")

    # BATCH_SIZE = 32
    # MAX_SEQ_LENGTH = 256
    # VOCAB_SIZE = 10000
    EPOCHS = 10
    LEARNING_RATE = 4e-4  # LLaDA recommended starting LR
    # N_SUP = 8             # Deep supervision steps (lowered from 16 for faster prototyping)
    MASK_TOKEN_ID = -100  # Our mathematical mask token

        # --- Hardware-Safe Hyperparameters for 4GB VRAM ---
    BATCH_SIZE = 4           # Dropped from 32
    GRADIENT_ACCUMULATION = 8 # Simulates a batch size of 32 (4 * 8)
    MAX_SEQ_LENGTH = 128      # Keep stories short for the prototype
    VOCAB_SIZE = 10000

    # TRM Dimensions
    D_MODEL = 128             # Dropped from 256
    N_HEADS = 4               
    D_FF = 512                # Dropped from 1024
    NUM_LAYERS = 2            

    # Recursion Limits (CRITICAL FOR VRAM)
    N_SUP = 4                 # Deep supervision steps (Max 4 for 4GB VRAM)
    N_RECURSIONS = 3          # The 'n' in latent_recursion (pass this into your functions)

    # ==========================================
    # 2. Load Data & Tokenizer
    # ==========================================
    train_loader, val_loader, tokenizer = get_dataloaders(
        batch_size=BATCH_SIZE, 
        max_length=MAX_SEQ_LENGTH, 
        vocab_size=VOCAB_SIZE
    )

    # ==========================================
    # 3. Initialize Model & Optimizer
    # ==========================================
    model = TRM_Diffusion(
        vocab_size=VOCAB_SIZE,
        d_model=256,       # Scaled down for TinyStories
        n_heads=8,
        d_ff=1024,
        num_layers=2,      # TRM paper found 2 layers optimal!
        mask_token_id=MASK_TOKEN_ID,
        n=N_RECURSIONS
    ).to(device)

    # AdamW with weight decay (standard for transformers)
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.1)

    # ==========================================
    # 4. Main Training Loop
    # ==========================================
    for epoch in range(EPOCHS):
        model.train()
        total_epoch_loss = 0
        
        # Wrap our dataloader in tqdm for a nice progress bar
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
        
        for prompts, targets in pbar:
            # Move data to GPU/MPS
            prompts = prompts.to(device)
            targets = targets.to(device)
            
            # Execute the deep supervision LLaDA training step
            loss = train_step_conditional(model, optimizer, prompts, targets, N_sup=N_SUP)
            
            total_epoch_loss += loss
            pbar.set_postfix({"Loss": f"{loss:.4f}"})
            
        avg_loss = total_epoch_loss / len(train_loader)
        print(f"\nEpoch {epoch+1} completed. Average Loss: {avg_loss:.4f}")
        
        # ==========================================
        # 5. Save Checkpoint
        # ==========================================
        checkpoint_path = f"trm_diffusion_epoch_{epoch+1}.pt"
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': avg_loss,
        }, checkpoint_path)
        print(f"Saved checkpoint to {checkpoint_path}")

        # ==========================================
        # 6. Live Generation Test (Sanity Check)
        # ==========================================
        print("\nSampling a story generation to check progress...")
        
        # Let's give it a 5-token prompt matching our training setup
        sample_text = "Once upon a time there"
        prompt_ids = tokenizer.encode(sample_text).ids
        
        # Pad the rest of the sequence with MASK tokens so it totals 256 length
        answer_length = MAX_SEQ_LENGTH - len(prompt_ids)
        mask_padding = [MASK_TOKEN_ID] * answer_length
        
        # This is x_t at timestep 1.0 (clean prompt + fully masked response)
        x_t_initial = torch.tensor([prompt_ids + mask_padding], dtype=torch.long, device=device)
        
        # Run diffusion generation (you will need to update the generate() function 
        # to accept x_t_initial instead of a separate prompt and answer!)
        generated_ids = generate_conditional(model, x_t_initial, prompt_length=len(prompt_ids))

if __name__ == "__main__":
    train()

In [ ]:
import torch
import torch.nn.functional as F
from torch.optim import AdamW
from torch.amp import GradScaler, autocast
from tqdm import tqdm

def train_4gb_vram():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training on: {device} with VRAM protections enabled.")

    # ==========================================
    # 1. Hardware-Safe Hyperparameters (4GB VRAM)
    # ==========================================
    BATCH_SIZE = 128            # Fits in 4GB VRAM
    GRADIENT_ACCUMULATION = 1 # Simulates batch size of 32 (4 * 8)
    MAX_SEQ_LENGTH = 1024      # Shorter stories for prototyping
    VOCAB_SIZE = 10000
    MASK_TOKEN_ID = -100
    
    # Tiny TRM Dimensions
    D_MODEL = 512             
    N_HEADS = 8               
    D_FF = 2048                
    NUM_LAYERS = 2            
    
    # VRAM-Safe Recursion Limits
    N_SUP = 16                 # Reduced deep supervision steps
    N_RECURSIONS = 6          # Reduced 'n' latent recursions
    EPOCHS = 3  

    # ==========================================
    # 2. Setup Data, Model, and Optimizers
    # ==========================================
    # (Assuming get_dataloaders is defined previously)
    train_loader, val_loader, tokenizer = get_dataloaders(
        batch_size=BATCH_SIZE, 
        max_length=MAX_SEQ_LENGTH, 
        vocab_size=VOCAB_SIZE
    )

    model = TRM_Diffusion(
        vocab_size=VOCAB_SIZE,
        d_model=D_MODEL,
        n_heads=N_HEADS,
        d_ff=D_FF,
        num_layers=NUM_LAYERS,
        mask_token_id=MASK_TOKEN_ID,
        n=N_RECURSIONS
    ).to(device)

    # Compile the model for H100 Tensor Cores
    print("Compiling model (this takes a minute on startup)...")
    model = torch.compile(model, mode="max-autotune")

    optimizer = AdamW(model.parameters(), lr=4e-4, weight_decay=0.1)
    scaler = GradScaler(device='cuda') # For Mixed Precision (FP16)

    # ==========================================
    # 3. The Unified Training Loop
    # ==========================================
    for epoch in range(EPOCHS):
        model.train()
        total_epoch_loss = 0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
        optimizer.zero_grad()
        
        for batch_idx, (prompts, targets) in enumerate(pbar):
            # Move targets to GPU (targets contain the full padded sequence)
            targets = targets.to(device)
            batch_size, seq_len = targets.size()
            
            # Assume first 3 tokens act as the prompt for conditional training
            prompt_length = 3 
            
            # --- START INLINED LOSS & FORWARD PASS ---
            # Sample continuous timestep t ~ U(0,1]
            t = torch.rand(1).item()
            t = max(t, 0.05)
            
            # Create a mask that ONLY applies to the response portion
            mask_probs = torch.zeros((batch_size, seq_len), device=device)
            mask_probs[:, prompt_length:] = t  
            mask_boolean = torch.bernoulli(mask_probs).bool()
            
            # Check if we actually masked anything this batch to prevent NaNs
            if mask_boolean.sum() == 0:
                continue

            # Create x_t (Clean prompt + Masked response)
            x_t = targets.clone()
            x_t[mask_boolean] = MASK_TOKEN_ID
            
            # Autocast automatically downcasts layers to FP16 to save VRAM
            with autocast(device_type='cuda', dtype=torch.bfloat16):
                # Get sequence embeddings and initialize latent z
                x_t_emb = model.get_embeddings(x_t)
                z = model.z_init.expand(batch_size, seq_len, model.d_model)
                
                batch_loss = 0.0
                
                # Deep Supervision Loop
                for step in range(N_SUP):
                    # Pass specific N_RECURSIONS to avoid memory blowup
                    logits, z = model.deep_recursion(x_t_emb, z)
                    
                    logits_flat = logits.view(-1, VOCAB_SIZE)
                    targets_flat = targets.view(-1)
                    mask_flat = mask_boolean.view(-1)
                    
                    # Compute cross-entropy strictly on the masked tokens
                    step_loss = F.cross_entropy(
                        logits_flat[mask_flat], 
                        targets_flat[mask_flat], 
                        reduction='mean'
                    )
                    
                    # Scale by 1/t (LLaDA objective)
                    scaled_step_loss = step_loss / t
                    batch_loss = batch_loss + scaled_step_loss
                
                # Average the loss over the supervision steps
                batch_loss = batch_loss / N_SUP
                
                # Divide by accumulation steps so gradients sum up to the scale of 1 full batch
                scaled_accum_loss = batch_loss / GRADIENT_ACCUMULATION
            
            # --- END INLINED LOSS & FORWARD PASS ---

            # Gradients accumulate safely in the background
            scaler.scale(scaled_accum_loss).backward()
            
            # Track loss for progress bar (multiply back by accumulation for accurate display)
            total_epoch_loss += scaled_accum_loss.item() * GRADIENT_ACCUMULATION
            
            # Optimizer Step (Every 8 batches or at the very end of the loader)
            if ((batch_idx + 1) % GRADIENT_ACCUMULATION == 0) or ((batch_idx + 1) == len(train_loader)):
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                
            pbar.set_postfix({"Loss": f"{(total_epoch_loss / (batch_idx + 1)):.4f}"})

        # ==========================================
        # 4. End of Epoch Actions
        # ==========================================
        print(f"Epoch {epoch+1} Avg Loss: {total_epoch_loss / len(train_loader):.4f}")
        torch.save(model.state_dict(), f"trm_tiny_epoch_{epoch+1}.pt")
        
        print("\nSampling generation...")
        
        # Give it a 3-word prompt to match the prompt_length
        prompt_text = "The little dog" 
        prompt_ids = tokenizer.encode(prompt_text).ids
        
        answer_length = MAX_SEQ_LENGTH - len(prompt_ids)
        mask_padding = [MASK_TOKEN_ID] * answer_length
        x_t_initial = torch.tensor([prompt_ids + mask_padding], dtype=torch.long, device=device)
        
        generated_ids = generate_conditional(
            model, 
            x_t_initial, 
            prompt_length=len(prompt_ids), 
            sampling_steps=32
        )
        
        eos_id = tokenizer.token_to_id("[EOS]")
        gen_list = [idx for idx in generated_ids[0].tolist() if idx not in (MASK_TOKEN_ID, eos_id)]
        print(f"Output:\n{tokenizer.decode(gen_list)}\n{'-'*60}\n")

if __name__ == "__main__":
    train_4gb_vram()

Training on: cuda with VRAM protections enabled.
Downloading/Loading TinyStories...


Epoch 1/10:  47%|████▋     | 251218/529929 [7:53:51<47:43:43,  1.62it/s, Loss=8.3418]

For H100

In [ ]:
import torch
import torch.nn.functional as F
from torch.optim import AdamW
from torch.amp import GradScaler, autocast
from tqdm import tqdm

def train_h100_act():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training on: {device} | H100 Optimizations + ACT Enabled")

    # ==========================================
    # 1. H100 80GB Hyperparameters
    # ==========================================
    BATCH_SIZE = 128            
    GRADIENT_ACCUMULATION = 1 
    MAX_SEQ_LENGTH = 1024      
    VOCAB_SIZE = 10000
    MASK_TOKEN_ID = -100
    
    # TRM Dimensions
    D_MODEL = 512             
    N_HEADS = 8               
    D_FF = 2048                
    NUM_LAYERS = 2            
    
    # Recursion & ACT Limits
    N_SUP = 16                 
    N_RECURSIONS = 6          
    EPOCHS = 10  
    TAU = 0.01 # The ACT Ponder Cost Penalty (Start small)

    # ==========================================
    # 2. Setup Data, Model, and Optimizers
    # ==========================================
    train_loader, val_loader, tokenizer = get_dataloaders(
        batch_size=BATCH_SIZE, 
        max_length=MAX_SEQ_LENGTH, 
        vocab_size=VOCAB_SIZE
    )

    model = TRM_Diffusion(
        vocab_size=VOCAB_SIZE,
        d_model=D_MODEL,
        n_heads=N_HEADS,
        d_ff=D_FF,
        num_layers=NUM_LAYERS,
        mask_token_id=MASK_TOKEN_ID
    ).to(device)

    print("Compiling model for Tensor Cores...")
    model = torch.compile(model, mode="max-autotune")

    optimizer = AdamW(model.parameters(), lr=4e-4, weight_decay=0.1)
    scaler = GradScaler(device='cuda')

    # ==========================================
    # 3. The ACT Training Loop
    # ==========================================
    for epoch in range(EPOCHS):
        model.train()
        total_epoch_loss = 0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
        optimizer.zero_grad()
        
        for batch_idx, (_, targets) in enumerate(pbar):
            targets = targets.to(device)
            batch_size, seq_len = targets.size()
            
            prompt_length = 3 
            
            t = torch.rand(1).item()
            t = max(t, 0.05)
            
            mask_probs = torch.zeros((batch_size, seq_len), device=device)
            mask_probs[:, prompt_length:] = t  
            mask_boolean = torch.bernoulli(mask_probs).bool()
            
            if mask_boolean.sum() == 0:
                continue

            x_t = targets.clone()
            x_t[mask_boolean] = MASK_TOKEN_ID
            
            with autocast(device_type='cuda', dtype=torch.bfloat16):
                x_t_emb = model.get_embeddings(x_t)
                z = model.z_init.expand(batch_size, seq_len, model.d_model)
                
                batch_loss = 0.0
                ponder_cost = 0.0
                halting_probs_sum = torch.zeros(batch_size, 1, device=device, dtype=torch.bfloat16)
                
                # --- The Deep Supervision & ACT Loop ---
                for step in range(N_SUP):
                    logits, z = model.deep_recursion(x_t_emb, z, n=N_RECURSIONS)
                    
                    # 1. Calculate ACT Halting Probabilities
                    step_halt_prob = model.get_halting_prob(z)
                    halting_probs_sum = halting_probs_sum + step_halt_prob
                    
                    # 2. Calculate Standard Diffusion Loss
                    logits_flat = logits.view(-1, VOCAB_SIZE)
                    targets_flat = targets.view(-1)
                    mask_flat = mask_boolean.view(-1)
                    
                    step_text_loss = F.cross_entropy(
                        logits_flat[mask_flat], 
                        targets_flat[mask_flat], 
                        reduction='mean'
                    )
                    
                    # Scale by 1/t
                    scaled_text_loss = step_text_loss / t
                    batch_loss = batch_loss + scaled_text_loss
                    
                    # 3. Calculate Ponder Cost
                    # Multiply the halting prob by the step number (penalizing late halting)
                    ponder_cost = ponder_cost + (step_halt_prob * (step + 1))
                
                # Average losses
                batch_loss = batch_loss / N_SUP
                avg_ponder_cost = ponder_cost.mean()
                
                # Combine Text Loss and Ponder Cost
                total_step_loss = batch_loss + (TAU * avg_ponder_cost)
                scaled_accum_loss = total_step_loss / GRADIENT_ACCUMULATION
            
            scaler.scale(scaled_accum_loss).backward()
            
            total_epoch_loss += scaled_accum_loss.item() * GRADIENT_ACCUMULATION
            
            if ((batch_idx + 1) % GRADIENT_ACCUMULATION == 0) or ((batch_idx + 1) == len(train_loader)):
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                
            pbar.set_postfix({
                "Loss": f"{(total_epoch_loss / (batch_idx + 1)):.4f}",
                "Ponder": f"{avg_ponder_cost.item():.2f}"
            })

        # ==========================================
        # 4. End of Epoch Actions
        # ==========================================
        print(f"Epoch {epoch+1} Avg Loss: {total_epoch_loss / len(train_loader):.4f}")
        torch.save(model.state_dict(), f"trm_h100_act_epoch_{epoch+1}.pt")
        
        print("\nSampling generation (Forcing full 16 steps)...")
        
        prompt_text = "The little dog" 
        prompt_ids = tokenizer.encode(prompt_text).ids
        
        answer_length = MAX_SEQ_LENGTH - len(prompt_ids)
        mask_padding = [MASK_TOKEN_ID] * answer_length
        x_t_initial = torch.tensor([prompt_ids + mask_padding], dtype=torch.long, device=device)
        
        # Generation runs without dynamic halting, as discussed!
        generated_ids = generate_conditional(
            model, 
            x_t_initial, 
            prompt_length=len(prompt_ids), 
            sampling_steps=32
        )
        
        eos_id = tokenizer.token_to_id("[EOS]")
        gen_list = [idx for idx in generated_ids[0].tolist() if idx not in (MASK_TOKEN_ID, eos_id)]
        print(f"Output:\n{tokenizer.decode(gen_list)}\n{'-'*60}\n")

if __name__ == "__main__":
    train_h100_act()